# WebInject2026 — build, validate, benchmark

This notebook orchestrates the v3 pipeline. All logic lives in the importable `src/` package so it is testable and reproducible; the notebook just runs it and shows results. See `DATASHEET.md` for the full description and `CHANGELOG.md` for what changed.

```
src/config.py     35 endpoints, 14 families, volumes, four anti-leak invariants
src/vocab.py      three-tier benign generators, ~2.3k groupable benign cores, injection guard
src/payloads.py   14 attack families: seed + grammar (templates) + chained evasions
src/build.py      synthesise -> (optional capture) -> normalise -> dedup -> payload + structural splits -> sessions
src/validate.py   26-check quality gate
src/baselines.py  reference results: random / payload-disjoint / structural, prevalence-corrected precision
src/evaluate.py   protocols (disjoint/structural/LOFO/evasion) + submission scorer
```

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
from src import config as C
C.validate_config()
print('config OK  version', C.VERSION)

## 1. Build

Offline synthesis is the default and needs no containers (reproducible from `seed=42`). To capture against live DVWA / Juice Shop / bWAPP instead, run `python -m src.build --capture` from a shell with the three apps running.

In [ ]:
!python -m src.build

In [ ]:
import pandas as pd
df = pd.read_csv('../data/webinject2026.csv')
print(df.shape)
print(json.load(open('../data/manifest.json'))['tier'])
df.sample(4, random_state=0)[['app','method','tier','family','placement','payload']]

## 2. Validate

The gate fails (non-zero exit) if any known shortcut or leak reappears. This is the guard that keeps the corpus 'no-flaws' across regenerations.

In [ ]:
!python -m src.validate

## 3. Benchmark

Reference baselines on both evaluation protocols. The random-vs-disjoint gap is the headline generalisation result: a model that only memorises payloads scores high on random and drops sharply on disjoint.

In [ ]:
!python -m src.baselines

## 4. Evaluation protocols

Beyond the headline split, `src/evaluate.py` runs the harder generalisation protocols — structural (template-disjoint), leave-one-family-out (zero-shot), and evasion hold-out — and reports precision corrected to realistic attack prevalences. Use `--predictions` to score your own model against the fixed test split.

In [ ]:
!python -m src.evaluate